# 09 — Deep learning (TEA): CNN própria e ResNet50 (PyTorch, protocolo revisado)

**Por que PyTorch nos dois projetos.** Mantém o mesmo código, o mesmo protocolo e as mesmas métricas nas duas bases. Isso melhora a consistência da implementação, mas **não** é afirmado como causa de nenhum resultado anterior: o desempenho fraco da CNN no TEA (AUC ≈ 0,49) pode vir dos dados, do tamanho da amostra, do efeito de site ou da implementação, e só a comparação empírica (célula de falsificação, opcional) separa essas hipóteses.

**Substitui** o notebook 09 em Keras/TensorFlow. Os resultados anteriores (CNN AUC 0,487; ResNet50 AUC 0,551) ficam como histórico; não os compare diretamente com os novos sem lembrar que o protocolo mudou (validação interna agrupada por site, normalização só no treino, limiar interno, sementes).

**Protocolo (idêntico nas duas bases)**
- Mesmos 5 folds externos agrupados por site do notebook 07. Nada do fold externo de teste é usado para treinar, normalizar, parar o treino ou escolher limiar.
- **Normalização** (média e desvio escalares do Fisher z) calculada só com o treino do fold externo.
- **Validação interna** agrupada por site (a mesma checagem de viabilidade do notebook 08) para early stopping (AUC) e para escolher o limiar de decisão. Se não houver partição viável, o treino usa número fixo de épocas e limiar 0,5, e isso é registrado.
- **CNN própria** (referência): 3 blocos conv 16/32/64 + BatchNorm + ReLU, pooling adaptativo e MLP. Treinada do zero.
- **ResNet50** (ImageNet) com backbone **congelado** e só a camada final treinada. Como o backbone não muda, as 2048 características são extraídas uma vez por fold e a camada final é treinada sobre elas (equivalente matemático, bem mais rápido). O benefício da ResNet50 sobre a CNN própria tem de aparecer nos resultados; não é presumido.
- Alertas automáticos: previsão de uma única classe, AUC de validação interna ≤ 0,5, teste de overfit antes da validação cruzada.
- Várias sementes por fold; o OOF salvo usa a primeira semente; a tabela longa guarda todas.
- Os folds são validação cruzada por site, **não** um conjunto de teste independente. Matrizes de conectividade não são imagens naturais: o uso da ResNet50 é exploratório.

In [2]:
# 1. Projeto, módulo de avaliação e dataset (TEA / ABIDE I)
import sys, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

_raiz = Path.cwd().resolve()
for _c in (_raiz, *_raiz.parents):
    if (_c / 'src' / 'config.py').exists():
        _raiz = _c
        break
else:
    raise RuntimeError('Não encontrei src/config.py. Abra a pasta TCC_TEA_TDAH no VS Code.')

if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src import avaliacao as av

criar_pastas()

NOME_BASE = 'TEA'
ROTULO_CASO = 'TEA'
ARQ_DATASET = PATHS['processed'] / 'dataset_abide_processado.pkl'
PASTA_RES = Path(PATHS['results'])
PASTA_MOD = Path(PATHS['models'])
PASTA_RES.mkdir(parents=True, exist_ok=True)
PASTA_MOD.mkdir(parents=True, exist_ok=True)

# 2. Parâmetros desta execução
import copy
import subprocess

try:
    import torch
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "torch", "torchvision"
    ])
    import torch

import torch.nn as nn
import torch.nn.functional as F

SEMENTES = [42, 43, 44]
EPOCAS_MAX_CNN = 60
EPOCAS_MAX_HEAD = 200
EPOCAS_FIXAS = 30
PACIENCIA = 10
BATCH = 32
LR_CNN, WD_CNN = 1e-3, 1e-4
LR_HEAD, WD_HEAD = 1e-3, 1e-2
RODAR_RESNET = True
USAR_PESOS_IMAGENET = True
RODAR_FALSIFICACAO = False

if av.RAPIDO:
    SEMENTES, EPOCAS_MAX_CNN, EPOCAS_MAX_HEAD, EPOCAS_FIXAS, PACIENCIA = [42], 3, 10, 3, 2
    USAR_PESOS_IMAGENET = False
    print('*** TCC_RAPIDO=1: teste curto, sem pesos ImageNet ***')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
d = av.carregar_dataset(ARQ_DATASET)
N_ROIS = av.n_rois_de(d['X'].shape[1])

print(f'Base: {NOME_BASE} | X: {d["X"].shape} | ROIs: {N_ROIS} | dispositivo: {DEVICE}')
print('Classes:', pd.Series(d['y']).value_counts().sort_index().to_dict(), '| sites:', len(np.unique(d['sites'])))
print('Intervalo de X (Fisher z):', float(d['X'].min()), 'a', float(d['X'].max()))

# Reconstrução das matrizes simétricas (diagonal = 0, pois não é conexão)
_i, _j = np.triu_indices(N_ROIS, k=1)
MATS = np.zeros((len(d['X']), N_ROIS, N_ROIS), dtype=np.float32)
MATS[:, _i, _j] = d['X']
MATS[:, _j, _i] = d['X']
_OFF = ~np.eye(N_ROIS, dtype=bool)
print('Matrizes:', MATS.shape)

  Using cached torch-2.14.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (37 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.0.3.0-py2.py3-none-any.whl.metadata (17 kB)
  Using cached nvidia_cudnn_cu13-9.24.0.43-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.30.7-py3-none-manylinux_2_18_x86_64.whl.metadata (2.1 kB)
  Using cached nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.1 kB)
  Using cached triton-3.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (978 bytes)
  Using cached nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cuda_nvrtc-13.0.88-py3-none-many

In [3]:
# 3. Modelos e funções de treino
class CNNConectividade(nn.Module):
    def __init__(self):
        super().__init__()
        def bloco(i, o):
            return [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True)]
        self.features = nn.Sequential(*bloco(1, 16), nn.MaxPool2d(2), *bloco(16, 32), nn.MaxPool2d(2),
                                      *bloco(32, 64), nn.AdaptiveAvgPool2d(1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.40), nn.Linear(64, 32),
                                        nn.ReLU(inplace=True), nn.Dropout(0.25), nn.Linear(32, 1))
    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


def estatisticas_treino(idx_treino):
    """Média e desvio escalares do Fisher z, só com o treino e fora da diagonal."""
    v = MATS[idx_treino][:, _OFF]
    return float(v.mean()), float(v.std() + 1e-8)


def normalizar(mats, mu, sd):
    out = (mats - mu) / sd
    out[:, np.arange(N_ROIS), np.arange(N_ROIS)] = 0.0
    return out.astype(np.float32)


@torch.no_grad()
def prever(modelo, X, batch=BATCH):
    modelo.eval()
    saidas = []
    for k in range(0, len(X), batch):
        xb = torch.as_tensor(X[k:k + batch]).to(DEVICE)
        saidas.append(torch.sigmoid(modelo(xb).reshape(-1)).cpu().numpy())
    return np.concatenate(saidas).astype(np.float32)


def treinar(modelo, Xtr, ytr, Xva=None, yva=None, epocas_max=60, lr=1e-3, wd=1e-4, paciencia=10,
            batch=BATCH, semente=42, epocas_fixas=EPOCAS_FIXAS):
    """Treina com BCE ponderada. Com validação interna: early stopping por AUC. Sem: épocas fixas."""
    g = torch.Generator().manual_seed(semente)
    modelo = modelo.to(DEVICE)
    n_pos, n_neg = int((ytr == 1).sum()), int((ytr == 0).sum())
    pos_w = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=DEVICE)
    perda_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=lr, weight_decay=wd)
    tem_val = Xva is not None and len(np.unique(yva)) == 2
    epocas = epocas_max if tem_val else epocas_fixas
    melhor_auc, melhor_estado, sem_melhora, hist = -1.0, None, 0, []
    Xt, yt = torch.as_tensor(Xtr), torch.as_tensor(ytr, dtype=torch.float32)
    for ep in range(epocas):
        modelo.train()
        ordem = torch.randperm(len(Xt), generator=g)
        soma = 0.0
        for k in range(0, len(Xt), batch):
            b = ordem[k:k + batch]
            if len(b) < 2:      # BatchNorm exige >= 2 amostras
                continue
            xb, yb = Xt[b].to(DEVICE), yt[b].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            perda = perda_fn(modelo(xb).reshape(-1), yb)
            perda.backward()
            opt.step()
            soma += perda.item() * len(b)
        auc_val = np.nan
        if tem_val:
            auc_val = av.roc_auc_score(yva, prever(modelo, Xva))
            if auc_val > melhor_auc + 1e-4:
                melhor_auc, melhor_estado, sem_melhora = auc_val, copy.deepcopy(modelo.state_dict()), 0
            else:
                sem_melhora += 1
        hist.append((soma / len(Xt), auc_val))
        if tem_val and sem_melhora >= paciencia:
            break
    if tem_val and melhor_estado is not None:
        modelo.load_state_dict(melhor_estado)
    return modelo, dict(epocas_rodadas=len(hist), auc_val_interna=(melhor_auc if tem_val else np.nan), hist=hist)


def escolher_limiar(yva, pva):
    """Limiar que maximiza a acurácia balanceada na validação INTERNA (0,5 se não houver validação)."""
    if yva is None or len(np.unique(yva)) < 2:
        return 0.5
    candidatos = np.unique(np.round(pva, 4))
    melhores = [(av.balanced_accuracy_score(yva, (pva >= t).astype(int)), -abs(t - 0.5), t) for t in candidatos]
    return float(max(melhores)[2])


def carregar_backbone_resnet():
    import torchvision
    from torchvision.models import resnet50, ResNet50_Weights
    pesos = ResNet50_Weights.DEFAULT if USAR_PESOS_IMAGENET else None
    try:
        m = resnet50(weights=pesos)
    except Exception as exc:
        raise RuntimeError('Não consegui obter os pesos ImageNet da ResNet50 (precisa de internet na primeira vez). '
                           'Conecte-se ou, só para teste, defina USAR_PESOS_IMAGENET=False.') from exc
    m.fc = nn.Identity()
    for p in m.parameters():
        p.requires_grad = False
    return m.eval().to(DEVICE)


@torch.no_grad()
def features_resnet(backbone, mats_norm, batch=16):
    """Matriz padronizada -> [0,1] (corte em ±3 desvios) -> 3 canais 224x224 -> normalização ImageNet."""
    media = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
    dp = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
    saidas = []
    for k in range(0, len(mats_norm), batch):
        x = torch.as_tensor(mats_norm[k:k + batch])[:, None].to(DEVICE)
        x = (x.clamp(-3, 3) + 3) / 6
        x = F.interpolate(x.repeat(1, 3, 1, 1), size=(224, 224), mode='bilinear', align_corners=False)
        saidas.append(backbone((x - media) / dp).cpu().numpy())
    return np.concatenate(saidas).astype(np.float32)


def padronizar_features(F_all, idx_treino):
    mu, sd = F_all[idx_treino].mean(0), F_all[idx_treino].std(0) + 1e-6
    return ((F_all - mu) / sd).astype(np.float32)

In [4]:
# 4. Teste de overfit (sanidade): a CNN deve conseguir memorizar 48 participantes
# Se a perda não cair bem, há erro de código/formato antes de qualquer validação cruzada.
rng = np.random.default_rng(SEED)
sel = rng.choice(len(MATS), size=48, replace=False)
if len(np.unique(d['y'][sel])) < 2:
    sel[:2] = [np.where(d['y'] == 0)[0][0], np.where(d['y'] == 1)[0][0]]
mu0, sd0 = estatisticas_treino(sel)
Xs = normalizar(MATS[sel], mu0, sd0)[:, None]
torch.manual_seed(SEED)
_, inf = treinar(CNNConectividade(), Xs, d['y'][sel], epocas_max=10 if av.RAPIDO else 80, lr=2e-3, wd=0, batch=16,
                 semente=SEED, epocas_fixas=10 if av.RAPIDO else 80)
print('Perda inicial: %.3f -> final: %.3f' % (inf['hist'][0][0], inf['hist'][-1][0]))
if not av.RAPIDO and inf['hist'][-1][0] > 0.5 * inf['hist'][0][0]:
    print('ALERTA: a perda não caiu pela metade ao memorizar 48 amostras — verifique o código antes de continuar.')
else:
    print('OK (ou modo rápido).')

Perda inicial: 0.683 -> final: 0.459
ALERTA: a perda não caiu pela metade ao memorizar 48 amostras — verifique o código antes de continuar.


In [5]:
# 5. Validação cruzada externa agrupada por site: CNN e ResNet50
print('FOLDS EXTERNOS'); display(av.resumo_folds(d))
folds = sorted(np.unique(d['fold_id']).tolist())
nomes = ['CNN'] + (['ResNet50'] if RODAR_RESNET else [])
proba_oof = {m: np.full(len(d['y']), np.nan, dtype=np.float32) for m in nomes}
pred_oof = {m: np.full(len(d['y']), -1, dtype=np.int8) for m in nomes}
linhas = []
backbone = carregar_backbone_resnet() if RODAR_RESNET else None
(PASTA_MOD / 'dl_folds').mkdir(exist_ok=True)

for fold in folds:
    te = np.where(d['fold_id'] == fold)[0]
    tr = np.where(d['fold_id'] != fold)[0]
    splits, info = av.splits_internos(d['y'][tr], d['sites'][tr], k_max=3, seed=SEED)
    if info['viavel']:
        in_tr, in_va = tr[splits[0][0]], tr[splits[0][1]]       # 1ª divisão interna: treino interno / validação interna
    else:
        in_tr, in_va = tr, None
    print(f'\n=== Fold {fold}: treino={len(tr)} (interno: {len(in_tr)}, validação interna: {0 if in_va is None else len(in_va)}) teste={len(te)}')

    mu, sd = estatisticas_treino(tr)                             # só com o treino externo
    NORM = normalizar(MATS, mu, sd)                              # (N, n, n) já normalizado
    yva = None if in_va is None else d['y'][in_va]

    for sem in SEMENTES:
        # ---- CNN própria
        torch.manual_seed(sem * 100 + fold); np.random.seed(sem * 100 + fold)
        t0 = av.agora()
        cnn, inf = treinar(CNNConectividade(), NORM[in_tr][:, None], d['y'][in_tr],
                           None if in_va is None else NORM[in_va][:, None], yva,
                           epocas_max=EPOCAS_MAX_CNN, lr=LR_CNN, wd=WD_CNN, paciencia=PACIENCIA, semente=sem)
        lim = escolher_limiar(yva, None if in_va is None else prever(cnn, NORM[in_va][:, None]))
        p = prever(cnn, NORM[te][:, None]); pr = (p >= lim).astype(np.int8)
        lin = av.linha_resultado(NOME_BASE, 'CNN', fold, sem, d['y'][te], pr, p, limiar=lim,
                                 auc_val_interna=inf['auc_val_interna'], epocas=inf['epocas_rodadas'],
                                 k_interno=int(info['k_usado']), seg_total=round(av.agora() - t0, 1))
        linhas.append(lin)
        if sem == SEMENTES[0]:
            proba_oof['CNN'][te], pred_oof['CNN'][te] = p, pr
            torch.save(cnn.state_dict(), PASTA_MOD / 'dl_folds' / f'cnn_fold{fold}.pt')
        alerta = ' <-- ALERTA: uma única classe prevista' if lin['classe_unica_prevista'] else ''
        if not np.isnan(inf['auc_val_interna']) and inf['auc_val_interna'] <= 0.5:
            alerta += ' <-- ALERTA: AUC de validação interna <= 0,5'
        print(f"  CNN      sem={sem} AUC={lin['auc_roc']:.3f} BalAcc={lin['balanced_accuracy']:.3f} "
              f"AUCval={inf['auc_val_interna']:.3f} limiar={lim:.2f} épocas={inf['epocas_rodadas']}{alerta}")
        del cnn

    if RODAR_RESNET:
        FEAT = padronizar_features(features_resnet(backbone, NORM), tr)   # extraído uma vez por fold
        for sem in SEMENTES:
            torch.manual_seed(sem * 100 + fold)
            t0 = av.agora()
            head, inf = treinar(nn.Sequential(nn.Dropout(0.3), nn.Linear(FEAT.shape[1], 1), nn.Flatten(0)),
                                FEAT[in_tr], d['y'][in_tr], None if in_va is None else FEAT[in_va], yva,
                                epocas_max=EPOCAS_MAX_HEAD, lr=LR_HEAD, wd=WD_HEAD, paciencia=PACIENCIA, semente=sem,
                                batch=64)
            lim = escolher_limiar(yva, None if in_va is None else prever(head, FEAT[in_va]))
            p = prever(head, FEAT[te]); pr = (p >= lim).astype(np.int8)
            lin = av.linha_resultado(NOME_BASE, 'ResNet50', fold, sem, d['y'][te], pr, p, limiar=lim,
                                     auc_val_interna=inf['auc_val_interna'], epocas=inf['epocas_rodadas'],
                                     k_interno=int(info['k_usado']), seg_total=round(av.agora() - t0, 1))
            linhas.append(lin)
            if sem == SEMENTES[0]:
                proba_oof['ResNet50'][te], pred_oof['ResNet50'][te] = p, pr
            alerta = ' <-- ALERTA: uma única classe prevista' if lin['classe_unica_prevista'] else ''
            print(f"  ResNet50 sem={sem} AUC={lin['auc_roc']:.3f} BalAcc={lin['balanced_accuracy']:.3f} "
                  f"AUCval={inf['auc_val_interna']:.3f} limiar={lim:.2f}{alerta}")
    pd.DataFrame(linhas).to_csv(PASTA_RES / '09_resultados_long.csv', index=False)   # checkpoint

for m in nomes:
    if np.any(pred_oof[m] < 0) or not np.isfinite(proba_oof[m]).all():
        raise RuntimeError(f'Previsões OOF incompletas para {m}.')
df_long = pd.DataFrame(linhas); df_long['tipo'] = 'externo'

FOLDS EXTERNOS


,fold,n_treino,n_teste,controles_teste,casos_teste,controles_treino,casos_treino,sites_teste,sobreposicao_sites,duas_classes_no_teste
0,0,830,41,19,22,449,381,YALE,0,True
1,1,698,173,101,72,367,331,"KKI, OLIN, SBL, UM_1",0,True
2,2,706,165,91,74,377,329,"CALTECH, LEUVEN_1, LEUVEN_2, PITT, TRINITY",0,True
3,3,547,324,175,149,293,254,"CMU, NYU, OHSU, SDSU, STANFORD, UCLA_1",0,True
4,4,703,168,82,86,386,317,"MAX_MUN, UCLA_2, UM_2, USM",0,True


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/maria/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:03<00:00, 27.3MB/s]



=== Fold 0: treino=830 (interno: 672, validação interna: 158) teste=41
  CNN      sem=42 AUC=0.510 BalAcc=0.501 AUCval=0.529 limiar=0.49 épocas=14
  CNN      sem=43 AUC=0.481 BalAcc=0.517 AUCval=0.511 limiar=0.48 épocas=17
  CNN      sem=44 AUC=0.469 BalAcc=0.468 AUCval=0.494 limiar=0.46 épocas=21 <-- ALERTA: AUC de validação interna <= 0,5
  ResNet50 sem=42 AUC=0.682 BalAcc=0.645 AUCval=0.592 limiar=0.61
  ResNet50 sem=43 AUC=0.677 BalAcc=0.606 AUCval=0.616 limiar=0.64
  ResNet50 sem=44 AUC=0.512 BalAcc=0.550 AUCval=0.637 limiar=0.56

=== Fold 1: treino=698 (interno: 606, validação interna: 92) teste=173
  CNN      sem=42 AUC=0.516 BalAcc=0.522 AUCval=0.624 limiar=0.49 épocas=22
  CNN      sem=43 AUC=0.501 BalAcc=0.476 AUCval=0.561 limiar=0.46 épocas=23
  CNN      sem=44 AUC=0.509 BalAcc=0.524 AUCval=0.601 limiar=0.52 épocas=43
  ResNet50 sem=42 AUC=0.512 BalAcc=0.501 AUCval=0.576 limiar=0.38
  ResNet50 sem=43 AUC=0.502 BalAcc=0.513 AUCval=0.597 limiar=0.47
  ResNet50 sem=44 AUC=0.56

In [6]:
# 6. Resumo
res = av.resumir_resultados(df_long, d, proba_oof)
pd.set_option('display.width', 200)
print('MÉDIA ± DP ENTRE FOLDS (média das sementes dentro de cada fold)'); display(res['por_fold'])
print('\nMÉDIA PONDERADA PELO TAMANHO DO FOLD'); display(res['ponderado'])
print('\nAUC OOF (1ª semente) com IC 95% por bootstrap de sites'); display(res['auc_oof_ic'])
print('\nVARIAÇÃO ENTRE SEMENTES (dp da AUC por modelo e fold)')
display(df_long.groupby(['modelo', 'fold'])['auc_roc'].agg(['mean', 'std']).round(3))
print('Critério a priori: IC inferior > 0,5 => "acima do acaso" (não é prova). Sem Wilcoxon com 5 folds.')
print('Colapso (uma classe prevista) em', int(df_long['classe_unica_prevista'].sum()), 'de', len(df_long), 'execuções.')

MÉDIA ± DP ENTRE FOLDS (média das sementes dentro de cada fold)


accuracy         balanced_accuracy         precision          recall         specificity              f1         auc_roc        
             mean     std              mean     std      mean     std    mean     std        mean     std    mean     std    mean     std
modelo                                                                                                                                   
CNN        0.4936  0.0380            0.5038  0.0270    0.4790  0.0529  0.5773  0.1753      0.4304  0.1816  0.5083  0.0679  0.4930  0.0208
ResNet50   0.5368  0.0479            0.5426  0.0438    0.5347  0.1025  0.5103  0.1487      0.5750  0.1906  0.5018  0.0585  0.5525  0.0586


MÉDIA PONDERADA PELO TAMANHO DO FOLD


,modelo,accuracy_pond,balanced_accuracy_pond,precision_pond,recall_pond,specificity_pond,f1_pond,auc_roc_pond
0,CNN,0.4937,0.5048,0.4677,0.6179,0.3916,0.5216,0.4915
1,ResNet50,0.5308,0.5341,0.5030,0.5292,0.5389,0.5028,0.5410



AUC OOF (1ª semente) com IC 95% por bootstrap de sites


,modelo,auc,ic_inf,ic_sup,n_boot_validos
0,CNN,0.4913,0.4543,0.5304,2000
1,ResNet50,0.5251,0.4914,0.5742,2000



VARIAÇÃO ENTRE SEMENTES (dp da AUC por modelo e fold)


mean    std
modelo   fold              
CNN      0     0.486  0.021
         1     0.509  0.007
         2     0.477  0.025
         3     0.478  0.004
         4     0.514  0.011
ResNet50 0     0.624  0.097
         1     0.526  0.034
         2     0.544  0.026
         3     0.547  0.036
         4     0.523  0.039

Critério a priori: IC inferior > 0,5 => "acima do acaso" (não é prova). Sem Wilcoxon com 5 folds.
Colapso (uma classe prevista) em 0 de 30 execuções.


In [7]:
# 7. (Opcional) Falsificação: a escala de entrada explica o resultado fraco? Só CNN, 1 semente.
# Compara (a) Fisher z padronizado no treino (usado acima) com (b) a escala original do TEA, (r+1)/2 sem padronizar.
# Se as duas ficarem perto de 0,5, a hipótese "escala/framework" perde força; se (a) >> (b), ela ganha.
if RODAR_FALSIFICACAO:
    resultados = {'(a) Fisher z padronizado': [], '(b) (r+1)/2 sem padronizar': []}
    for fold in folds:
        te = np.where(d['fold_id'] == fold)[0]; tr = np.where(d['fold_id'] != fold)[0]
        splits, info = av.splits_internos(d['y'][tr], d['sites'][tr], k_max=3, seed=SEED)
        in_tr, in_va = (tr[splits[0][0]], tr[splits[0][1]]) if info['viavel'] else (tr, None)
        mu, sd = estatisticas_treino(tr)
        entradas = {'(a) Fisher z padronizado': normalizar(MATS, mu, sd),
                    '(b) (r+1)/2 sem padronizar': ((np.tanh(MATS) + 1) / 2).astype(np.float32)}
        for nome_ent, ENT in entradas.items():
            torch.manual_seed(SEED + fold)
            m, inf = treinar(CNNConectividade(), ENT[in_tr][:, None], d['y'][in_tr],
                             None if in_va is None else ENT[in_va][:, None], None if in_va is None else d['y'][in_va],
                             epocas_max=EPOCAS_MAX_CNN, lr=LR_CNN, wd=WD_CNN, paciencia=PACIENCIA, semente=SEED)
            resultados[nome_ent].append(av.roc_auc_score(d['y'][te], prever(m, ENT[te][:, None])))
    print(pd.DataFrame(resultados).round(3)); print(pd.DataFrame(resultados).mean().round(3))
else:
    print('Falsificação desligada (RODAR_FALSIFICACAO=False).')

Falsificação desligada (RODAR_FALSIFICACAO=False).


In [8]:
# 8. Salvar
df_long.to_csv(PASTA_RES / '09_resultados_long.csv', index=False)
res['por_fold'].to_csv(PASTA_RES / '09_resumo_por_fold.csv')
res['ponderado'].to_csv(PASTA_RES / '09_resumo_ponderado.csv', index=False)
res['auc_oof_ic'].to_csv(PASTA_RES / '09_auc_oof_ic.csv', index=False)
av.salvar_oof(PASTA_RES / '09_predicoes_oof.npz', d, proba_oof, pred_oof)
print('Salvo em:', PASTA_RES)

Salvo em: /home/maria/TCC_TEA/Results
